In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Qiskit & Qiskit Machine Learning
from qiskit.circuit.library import ZZFeatureMap, ZFeatureMap
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorSampler



In [ ]:

data = pd.read_csv("binding_dataset_transformed.csv")

data = data.fillna(0)

print("Datos procesados. Tamaño final:", data.shape)
data.head()

Datos procesados. Tamaño final: (80, 12)


,Binding Energy,Otsu Class theshold -77.8,Abs dist to threshold,a_1,a_2,a_3,a_4,a_5,a_6,a_7,a_8,a_9
0,-92.5,1,14.7,1,18,14,9,10,1,20,18,13
1,-77.1,0,0.7,18,14,9,10,1,20,18,13,6
2,-99.6,1,21.8,1,20,18,13,6,13,1,16,10
3,-48.5,0,29.3,14,20,14,14,5,1,5,0,10
4,-50.1,0,27.7,10,10,15,15,11,20,15,18,0


In [ ]:

X = data.drop(columns=['Otsu Class theshold -77.8']).values
y = data['Otsu Class theshold -77.8'].values


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle=True, random_state=42
)


scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dimentions of training: {X_train_scaled.shape}")
print(f"Dimentions of test: {X_test_scaled.shape}")

Dimensiones de entrenamiento: (56, 11)
Dimensiones de prueba: (24, 11)


In [7]:
svm_classic = SVC(kernel='rbf', C=1.0)
svm_classic.fit(X_train_scaled, y_train)
y_pred_classic = svm_classic.predict(X_test_scaled)

print("--- SVM (RBF) ---")
print(classification_report(y_test, y_pred_classic))

--- SVM (RBF) ---
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        16
           1       0.88      0.88      0.88         8

    accuracy                           0.92        24
   macro avg       0.91      0.91      0.91        24
weighted avg       0.92      0.92      0.92        24



In [8]:

num_features = X_train_scaled.shape[1]
feature_map = ZZFeatureMap(feature_dimension=num_features, reps=1)


sampler = StatevectorSampler() 

quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)


qsvc = QSVC(quantum_kernel=quantum_kernel)

print("Training QSVC....")
qsvc.fit(X_train_scaled, y_train)

print("Training compleated")

Training QSVC....


KeyboardInterrupt: 

In [ ]:

train_score = qsvc.score(X_train_scaled, y_train)
test_score = qsvc.score(X_test_scaled, y_test)

print(f"Accuracy in training: {train_score:.2f}")
print(f"Accuracy in test: {test_score:.2f}")

y_pred_qsvc = qsvc.predict(X_test_scaled)
print("\n--- Report of Clasificatión of QSVC ---")
print(classification_report(y_test, y_pred_qsvc))

Accuracy en Entrenamiento: 1.00
Accuracy en Prueba: 0.33

--- Reporte de Clasificación QSVC ---
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        16
           1       0.33      1.00      0.50         8

    accuracy                           0.33        24
   macro avg       0.17      0.50      0.25        24
weighted avg       0.11      0.33      0.17        24



/home/LuisPabloElChido/miniconda3/envs/qiskit_general/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/LuisPabloElChido/miniconda3/envs/qiskit_general/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/LuisPabloElChido/miniconda3/envs/qiskit_general/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beh